# EF · 微调的价值 + HuggingFace vs unsloth 框架对比（Qwen3-0.6B）

> **罗氏 · 大模型微调课程实操**｜任务：临床短句 → 结构化 JSON 抽取
> 上层设计见 `../docs/2026-08-14-finetuning-course-design.md`

## 这个 notebook 回答两个问题
**① 微调到底值不值？**（头条）在**同一留出集**上比 3 种方案：
- **不微调·零样本**：直接给 base Qwen3-0.6B 指令
- **不微调·少样本(3例)**：prompt 里塞 3 个示例
- **微调后**：LoRA SFT 过的模型

关键看点：指令给了字段名，却**没说 `sex` 要归一成 `M/F`**（原文是 "female"）。base 只能猜 → 精确匹配大量失败；微调把这个隐含约定"内化"进权重 → 干净稳定。这正是 F1 的命题：**行为/格式用微调，而且能免掉长 prompt（降本降延迟）**。

**② 微调框架 HF vs unsloth 差多少？**（次要）同数据、同 LoRA 配置、同步数下比 **训练时间 / 峰值显存 / 收敛 / 上手成本**：
- **路线 A**：HuggingFace 生态 —— `transformers` + `trl(SFTTrainer)` + `peft(LoRA)`（看得见每一步，生态标准）
- **路线 B**：`unsloth` —— `FastLanguageModel`（内核优化，主打更快更省显存）

## 环境可移植性
- **目标环境**：AWS SageMaker Notebook，单卡 **T4（16G 显存）**
- **T4 硬约束**：数值精度用 **fp16**（不支持 bf16）；注意力用 **sdpa**（不支持 FlashAttention-2）
- 仅用开源框架，**不依赖 SageMaker SDK**，可在任意带 GPU（≥8G）环境运行
- 两条路线各写成独立脚本、**子进程运行**：显存测量干净，且 unsloth 的 monkey-patch 不会污染 HF 路线

## 运行顺序
1. `环境自检` → 2. `安装依赖`（首次，装完**重启 kernel**）→ 3. `生成数据` → 4. `共用配置`
→ 5. 写出 `train_hf.py` / `train_unsloth.py` → 6. 分别 `!python` 运行 → 7. 对比

## 1. 环境自检
检查 GPU、显存、bf16 支持（T4 应为 False，印证为何用 fp16）与各库版本。

In [ ]:
import platform, subprocess, torch
print("python", platform.python_version(), "| torch", torch.__version__,
      "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported(), " <- T4 应为 False => 用 fp16")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM total {total/1e9:.1f} GB | free {free/1e9:.1f} GB")
    try:
        print(subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"]).decode().strip())
    except Exception as e:
        print("nvidia-smi n/a:", e)
else:
    print("!! 未检测到 GPU，本 notebook 需要 GPU 环境")
for p in ["transformers", "trl", "peft", "datasets", "accelerate", "unsloth"]:
    try:
        m = __import__(p); print(p, getattr(m, "__version__", "?"))
    except Exception:
        print(p, "未安装")

## 2. 安装依赖（首次运行一次）
**内核提示**：优先选带 GPU 的 PyTorch 内核（如 SageMaker `conda_pytorch_p310`），它已自带 torch+CUDA，可注释掉下方 `torch` 那行。若用 `conda_python3` 这类不带 torch 的内核，保留 `torch` 安装。
⚠️ 装完 `unsloth` 后**重启 kernel** 再继续——unsloth 会调整 transformers/trl 版本。

In [ ]:
# 首次运行执行一次；装完 unsloth 后建议【重启 kernel】再往下跑（unsloth 会调整依赖版本）
# 若当前内核已自带 torch（如 SageMaker 的 conda_pytorch_p310 内核），可注释掉下一行
%pip install -q torch
%pip install -q -U transformers trl peft datasets accelerate
# unsloth：Colab 免费版即 T4，官方支持 T4。受限网络若失败见末尾 troubleshooting
%pip install -q -U unsloth

## 3. 生成数据集（HF / unsloth 共用）
合成「临床短句→JSON」抽取任务。**flag 直接写在句子里 → 这是纯格式/行为任务，不是知识注入**（呼应 F1：知识用 RAG，行为用微调）。真实场景请替换为脱敏后的自有数据。

In [ ]:
# 生成合成"临床短句 -> 结构化 JSON 抽取"数据集（HF 与 unsloth 共用）
# flag 字段直接写在句子里 => 纯格式转换任务(行为，非知识)，呼应 F1 头号命题
import json, random, os
random.seed(0)
os.makedirs("data", exist_ok=True)

SEX = ["male", "female"]
# (检验项, 单位, 取值范围, 正常范围)
TESTS = [
    ("hemoglobin", "g/dL", (6, 18), (12, 16)),
    ("glucose", "mg/dL", (50, 400), (70, 110)),
    ("creatinine", "mg/dL", (0.3, 8.0), (0.6, 1.3)),
    ("potassium", "mmol/L", (2.0, 7.0), (3.5, 5.1)),
    ("WBC", "10^9/L", (1.0, 30.0), (4.0, 11.0)),
]
INSTR = ("Extract the fields (age, sex, test, value, unit, flag) from the clinical "
         "note and return ONLY a JSON object.")

def make():
    age = random.randint(18, 89)
    sex = random.choice(SEX)
    name, unit, (lo, hi), (nlo, nhi) = random.choice(TESTS)
    val = round(random.uniform(lo, hi), 1)
    flag = "normal" if nlo <= val <= nhi else ("high" if val > nhi else "low")
    text = random.choice([
        f"A {age}-year-old {sex} patient had {name} measured at {val} {unit}, flagged as {flag}.",
        f"Lab report - {sex}, {age} yo: {name} = {val} {unit} ({flag}).",
        f"{name} for a {sex} aged {age} came back {val} {unit}, considered {flag}.",
    ])
    out = {"age": age, "sex": "M" if sex == "male" else "F", "test": name,
           "value": val, "unit": unit, "flag": flag}
    return {"input": INSTR + "\n\nNote: " + text,
            "output": json.dumps(out, ensure_ascii=False)}

rows = [make() for _ in range(240)]
train, ev = rows[:200], rows[200:]
for fn, part in [("data/train.jsonl", train), ("data/eval.jsonl", ev)]:
    with open(fn, "w") as f:
        for r in part:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("train:", len(train), "| eval:", len(ev))
print("--- 样例 ---")
print(train[0]["input"])
print("=>", train[0]["output"])

## 4. 共用配置
两条路线读同一份 `config.json`，保证 LoRA 秩、步数、学习率、batch 完全一致，对比才公平。`max_steps=60` 是快速冒烟值，正式训练可调大或改用 `num_train_epochs`。

In [ ]:
# 两条路线共用的配置（写盘，供 train_*.py 读取，保证对比公平）
import json
CFG = {
    "model_id": "Qwen/Qwen3-0.6B",
    "max_seq_length": 512,
    "lora_r": 16, "lora_alpha": 16,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    "batch_size": 4, "grad_accum": 2, "max_steps": 60,
    "lr": 2e-4, "warmup": 5, "seed": 3407, "n_eval": 6,
}
json.dump(CFG, open("config.json", "w"), indent=2)
print(CFG)

## 5. 基线：不微调直接用 base 模型
先量出「未微调」的水平（零样本 + 少样本），作为微调价值的对照。**在训练之前跑**。
看点：base 大概率能吐出结构大致对的 JSON，但会栽在**隐含约定**（`sex` 未归一成 M/F）和**输出干净度**（多余解释/围栏）上 → 精确匹配低。少样本能部分补救，但每次推理都要塞示例。

In [ ]:
%%writefile eval_base.py
# eval_base.py —— 基线：未微调的 base Qwen3-0.6B，零样本 & 少样本(3例) prompt
# 用来体现"微调的价值"：同一留出集、同样指令，只是没训练过。
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CFG = json.load(open("config.json"))
tok = AutoTokenizer.from_pretrained(CFG["model_id"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], torch_dtype=torch.float16, attn_implementation="sdpa").to("cuda")
model.eval()

eval_rows = [json.loads(l) for l in open("data/eval.jsonl")][:CFG["n_eval"]]
fewshot = [json.loads(l) for l in open("data/train.jsonl")][:3]  # 3 个示例

def gen(msgs):
    try:
        prompt = tok.apply_chat_template(msgs, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**ids, max_new_tokens=128, do_sample=False,
                           pad_token_id=tok.pad_token_id)
    return tok.decode(g[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse(s):
    try:
        i, j = s.find("{"), s.rfind("}")
        return json.loads(s[i:j + 1]) if i >= 0 and j > i else None
    except Exception:
        return None

def evaluate(build_msgs):
    gens = []
    for r in eval_rows:
        gens.append({"input": r["input"], "gold": r["output"], "pred": gen(build_msgs(r))})
    valid = sum(1 for x in gens if parse(x["pred"]) is not None)
    exact = sum(1 for x in gens if parse(x["pred"]) is not None and parse(x["pred"]) == parse(x["gold"]))
    return {"valid_json_rate": round(valid / len(gens), 3),
            "exact_match": round(exact / len(gens), 3), "samples": gens}

zeroshot = evaluate(lambda r: [{"role": "user", "content": r["input"]}])

def fs_msgs(r):
    m = []
    for ex in fewshot:
        m.append({"role": "user", "content": ex["input"]})
        m.append({"role": "assistant", "content": ex["output"]})
    m.append({"role": "user", "content": r["input"]})
    return m
fewshot_res = evaluate(fs_msgs)

res = {"framework": "base Qwen3-0.6B (未微调)",
       "zeroshot": zeroshot, "fewshot": fewshot_res}
json.dump(res, open("results_base.json", "w"), ensure_ascii=False, indent=2)
print("base 零样本 : validJSON %.0f%% | exact %.0f%%" % (
    zeroshot["valid_json_rate"] * 100, zeroshot["exact_match"] * 100))
print("base 少样本 : validJSON %.0f%% | exact %.0f%%" % (
    fewshot_res["valid_json_rate"] * 100, fewshot_res["exact_match"] * 100))


### 运行基线评估
首次会下载 Qwen3-0.6B（约 1.2GB）。

In [ ]:
!python eval_base.py

## 6A. 写出 HuggingFace 训练脚本
要点：fp16 + sdpa 加载；`SFTTrainer` + `peft` LoRA；末尾在留出集上生成并算合法 JSON 率 / 精确匹配。
本对比统一用**全序列 SFT**（两条路线一致、稳跑）。**completion-only loss masking**（只在回答段算 loss）需要 chat 模板含 `{% generation %}` 标记，Qwen3 默认没有，作为 F5 教学版的判质#2 单独讲，这里不启用。

In [ ]:
%%writefile train_hf.py
# train_hf.py —— HuggingFace 路线：transformers + trl(SFTTrainer) + peft(LoRA)
import json, time, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

CFG = json.load(open("config.json"))

tok = AutoTokenizer.from_pretrained(CFG["model_id"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# T4 关键：fp16 + sdpa（禁 bf16 / FlashAttention-2）
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"], torch_dtype=torch.float16, attn_implementation="sdpa",
).to("cuda")
model.config.use_cache = False

def to_messages(ex):
    return {"messages": [
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]}
train_ds = load_dataset("json", data_files="data/train.jsonl", split="train").map(
    to_messages, remove_columns=["input", "output"])

lora = LoraConfig(
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=0.0,
    target_modules=CFG["target_modules"], bias="none", task_type="CAUSAL_LM")

base = dict(
    output_dir="out_hf",
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    max_steps=CFG["max_steps"], learning_rate=CFG["lr"],
    warmup_steps=CFG["warmup"], logging_steps=5,
    fp16=True, bf16=False, optim="adamw_torch",
    lr_scheduler_type="linear", seed=CFG["seed"], report_to="none")

# 本对比 notebook 统一用【全序列 SFT】：两条路线一致、不依赖模板特性、稳跑。
# 说明：completion-only 掩码(只在回答段算 loss)需要 chat 模板含 {% generation %} 标记，
# Qwen3 默认模板没有，强开会在 tokenize 时报错。掩码留到 F5 教学版判质#2 单独讲。
sft_cfg = SFTConfig(**base)
print("[hf] loss = 全序列 SFT (completion-only 掩码见 F5 教学版)")

try:
    trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds,
                         peft_config=lora, processing_class=tok)
except TypeError:
    trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds,
                         peft_config=lora, tokenizer=tok)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
out = trainer.train()
train_time = time.time() - t0
peak = torch.cuda.max_memory_reserved() / 1e9

final_loss = None
for rec in reversed(trainer.state.log_history):
    if "loss" in rec:
        final_loss = rec["loss"]; break

# ---- 在留出集上生成，评估格式遵循 ----
model = trainer.model
model.config.use_cache = True
model.eval()
eval_rows = [json.loads(l) for l in open("data/eval.jsonl")][:CFG["n_eval"]]
gens = []
for r in eval_rows:
    msgs = [{"role": "user", "content": r["input"]}]
    try:
        prompt = tok.apply_chat_template(msgs, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**ids, max_new_tokens=128, do_sample=False,
                           pad_token_id=tok.pad_token_id)
    txt = tok.decode(g[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gens.append({"input": r["input"], "gold": r["output"], "pred": txt})

def parse(s):
    try:
        i, j = s.find("{"), s.rfind("}")
        return json.loads(s[i:j + 1]) if i >= 0 and j > i else None
    except Exception:
        return None
valid = sum(1 for x in gens if parse(x["pred"]) is not None)
exact = sum(1 for x in gens if parse(x["pred"]) is not None and parse(x["pred"]) == parse(x["gold"]))

res = {
    "framework": "huggingface (transformers+trl+peft)",
    "train_time_s": round(train_time, 1),
    "peak_vram_gb": round(peak, 2),
    "samples_per_s": round(out.metrics.get("train_samples_per_second", 0.0), 2),
    "final_loss": final_loss,
    "valid_json_rate": round(valid / len(gens), 3),
    "exact_match": round(exact / len(gens), 3),
    "samples": gens,
}
json.dump(res, open("results_hf.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in res.items() if k != "samples"}, ensure_ascii=False, indent=2))


## 6B. 写出 unsloth 训练脚本
要点：**`from unsloth import FastLanguageModel` 必须最先 import**（它 monkey-patch transformers）；`load_in_4bit=False` 用 16-bit LoRA 与 HF 对齐；其余训练/评估逻辑一致。import 失败会写错误到结果并优雅跳过（T4 上多为依赖/网络问题）。

In [ ]:
%%writefile train_unsloth.py
# train_unsloth.py —— unsloth 路线：FastLanguageModel + trl(SFTTrainer)
# 关键：unsloth 必须在 transformers 之前 import（它会 monkey-patch）
import json, time
try:
    from unsloth import FastLanguageModel
    UNSLOTH_OK, ERR = True, ""
except Exception as e:
    UNSLOTH_OK, ERR = False, repr(e)

import torch
CFG = json.load(open("config.json"))

if not UNSLOTH_OK:
    json.dump({"framework": "unsloth", "error": "import failed: " + ERR},
              open("results_unsloth.json", "w"), ensure_ascii=False, indent=2)
    print("[unsloth] import 失败：", ERR)
    print("  T4 上多为依赖/网络问题，见 notebook 末尾 troubleshooting。跳过本路线。")
    raise SystemExit(0)

# T4：fp16、16-bit LoRA（load_in_4bit=False，与 HF 对齐做公平对比）
model, tok = FastLanguageModel.from_pretrained(
    model_name=CFG["model_id"], max_seq_length=CFG["max_seq_length"],
    dtype=torch.float16, load_in_4bit=False)

model = FastLanguageModel.get_peft_model(
    model, r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=0.0,
    target_modules=CFG["target_modules"], bias="none",
    use_gradient_checkpointing=False,   # 与 HF 对齐；0.6B 无需
    random_state=CFG["seed"])

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

def to_messages(ex):
    return {"messages": [
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ]}
train_ds = load_dataset("json", data_files="data/train.jsonl", split="train").map(
    to_messages, remove_columns=["input", "output"])

base = dict(
    output_dir="out_unsloth",
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    max_steps=CFG["max_steps"], learning_rate=CFG["lr"],
    warmup_steps=CFG["warmup"], logging_steps=5,
    fp16=True, bf16=False, optim="adamw_torch",
    lr_scheduler_type="linear", seed=CFG["seed"], report_to="none")

# 与 HF 一致：全序列 SFT（不启用回答段掩码，避免模板依赖）
sft_cfg = SFTConfig(**base)
print("[unsloth] loss = 全序列 SFT")

try:
    trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds,
                         processing_class=tok)
except TypeError:
    trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=train_ds, tokenizer=tok)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
out = trainer.train()
train_time = time.time() - t0
peak = torch.cuda.max_memory_reserved() / 1e9

final_loss = None
for rec in reversed(trainer.state.log_history):
    if "loss" in rec:
        final_loss = rec["loss"]; break

FastLanguageModel.for_inference(model)
eval_rows = [json.loads(l) for l in open("data/eval.jsonl")][:CFG["n_eval"]]
gens = []
for r in eval_rows:
    msgs = [{"role": "user", "content": r["input"]}]
    try:
        prompt = tok.apply_chat_template(msgs, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**ids, max_new_tokens=128, do_sample=False,
                           pad_token_id=tok.pad_token_id)
    txt = tok.decode(g[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gens.append({"input": r["input"], "gold": r["output"], "pred": txt})

def parse(s):
    try:
        i, j = s.find("{"), s.rfind("}")
        return json.loads(s[i:j + 1]) if i >= 0 and j > i else None
    except Exception:
        return None
valid = sum(1 for x in gens if parse(x["pred"]) is not None)
exact = sum(1 for x in gens if parse(x["pred"]) is not None and parse(x["pred"]) == parse(x["gold"]))

res = {
    "framework": "unsloth",
    "train_time_s": round(train_time, 1),
    "peak_vram_gb": round(peak, 2),
    "samples_per_s": round(out.metrics.get("train_samples_per_second", 0.0), 2),
    "final_loss": final_loss,
    "valid_json_rate": round(valid / len(gens), 3),
    "exact_match": round(exact / len(gens), 3),
    "samples": gens,
}
json.dump(res, open("results_unsloth.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in res.items() if k != "samples"}, ensure_ascii=False, indent=2))


## 7A. 运行 HF 路线
首次会下载 Qwen3-0.6B（约 1.2GB）。

In [ ]:
# 子进程运行 HF 路线（隔离测量：干净的显存计数、不受 unsloth patch 影响）
!python train_hf.py

## 7B. 运行 unsloth 路线

In [ ]:
# 子进程运行 unsloth 路线（unsloth 在自己进程里 import，patch 不污染 HF 结果）
!python train_unsloth.py

## 8. 对比结果

In [ ]:
# 汇总对比：不微调(base) vs 微调(HF / unsloth)
import json
def load(p):
    try:
        return json.load(open(p))
    except FileNotFoundError:
        return None
base, hf, us = load("results_base.json"), load("results_hf.json"), load("results_unsloth.json")

rows = []  # 方案, 训练秒, 峰值显存, 末loss, 合法JSON率, 精确匹配
if base:
    rows.append(("base 零样本", "-", "-", "-",
                 base["zeroshot"]["valid_json_rate"], base["zeroshot"]["exact_match"]))
    rows.append(("base 少样本(3例)", "-", "-", "-",
                 base["fewshot"]["valid_json_rate"], base["fewshot"]["exact_match"]))
if hf and "error" not in hf:
    rows.append(("HF 微调", hf["train_time_s"], hf["peak_vram_gb"], hf["final_loss"],
                 hf["valid_json_rate"], hf["exact_match"]))
if us and "error" not in us:
    rows.append(("unsloth 微调", us["train_time_s"], us["peak_vram_gb"], us["final_loss"],
                 us["valid_json_rate"], us["exact_match"]))

hdr = ["方案", "训练秒", "峰值显存GB", "末loss", "合法JSON率", "精确匹配"]
tbl = [hdr] + [[str(x) for x in r] for r in rows]
w = [max(len(tbl[i][c]) for i in range(len(tbl))) for c in range(len(hdr))]
for i, row in enumerate(tbl):
    print(" | ".join(row[c].ljust(w[c]) for c in range(len(hdr))))
    if i == 0:
        print("-+-".join("-" * w[c] for c in range(len(hdr))))

print()
if base and hf and "error" not in hf:
    z = base["zeroshot"]["exact_match"] * 100
    print(f"★ 微调的价值：精确匹配 {z:.0f}% (base 零样本) -> {hf['exact_match']*100:.0f}% (微调后)")
if hf and us and "error" not in hf and "error" not in us and us.get("train_time_s"):
    print(f"★ 框架对比：unsloth 训练快 {hf['train_time_s']/us['train_time_s']:.2f}x | "
          f"峰值显存 {us['peak_vram_gb']/hf['peak_vram_gb']:.2f}x (越小越省)")
if us and "error" in us:
    print("unsloth 未运行：", us["error"])

### 并排看生成（格式遵循质量）

In [ ]:
# 并排看生成：base 零样本(未微调) vs 微调后 —— 直观看差在哪
import json
def load(p):
    try:
        return json.load(open(p))
    except FileNotFoundError:
        return None
base, hf = load("results_base.json"), load("results_hf.json")
bs = base["zeroshot"]["samples"] if base else []
hs = hf.get("samples", []) if hf else []
ref = hs if hs else bs
for i in range(max(len(bs), len(hs))):
    print("-" * 72)
    if i < len(ref):
        print("输入 :", ref[i]["input"].split("Note:")[-1].strip())
        print("gold :", ref[i]["gold"])
    if i < len(bs):
        print("base零样本 :", bs[i]["pred"])
    if i < len(hs):
        print("微调后     :", hs[i]["pred"])

## 9. 判质与观察引导（无标准答案，能说出理由即达标）

对着上面的数字与生成结果，回答：
1. **速度/显存**：unsloth 快多少、省多少？在 0.6B 这个小模型上差距明显吗？想想模型放大到 7B 时会怎样。
2. **收敛**：两条路线的末 loss 接近吗？若差得远，先怀疑什么（随机种子/数据顺序/patch 差异）？
3. **格式遵循**：合法 JSON 率和精确匹配谁高？看并排生成——错在哪（字段缺失/多字/flag 抄错/JSON 不闭合）？
4. **对比公平性自查**：这个对比可信吗？两条路线是否真的"只差框架"？（同数据/同配置/同步数/都关 GC/子进程隔离——还有什么没控住？warmup、cudnn 非确定性、下载的权重是否同一份？）
5. **上手成本**：读两个脚本，哪套代码量少、心智负担低？unsloth 省的那几行，代价是什么（灵活性/可调试性/版本绑定）？

> **易踩坑**：只看"末 loss 下降 / 合法 JSON 率高"就宣布成功，是本课判质#4 要打的靶子——
> 别忘了还要测**通用能力有没有退化（灾难性遗忘）**。本对比 notebook 只测了目标任务，完整评估在 F5 核心 lab 展开。

## 10. Troubleshooting（T4 常见问题）

- **unsloth import/安装失败**：多为依赖版本或网络。可只跑 HF 路线（对比 cell 会显示 unsloth error 并跳过）。
  受限网络下 `pip install unsloth` 可能拉不到 `unsloth_zoo`；确认出网或用离线 wheel。
- **CUDA OOM**：调小 `config.json` 的 `batch_size`（4→2/1）或 `max_seq_length`（512→256）；或把 `grad_accum` 调大保持有效 batch。
- **bf16 报错 / 训练 NaN**：确认用的是 **fp16 不是 bf16**（T4 不支持 bf16）。fp16 若偶发 NaN，降 `lr`（2e-4→1e-4）。
- **`no assistant tokens` / `{% generation %}` RuntimeError**：本 notebook 已统一用**全序列 SFT**、不启用 `assistant_only_loss`，不应再触发。若你看到这个报错，说明磁盘上跑的是**旧的 `train_hf.py`**——重跑 `%%writefile` 那格重新落盘（见下条）。
- **改了 notebook 但报错依旧**：`%%writefile` 生成的 `.py` 不会自动更新，必须**重跑写文件那格**。稳妥做法：`!rm -f train_hf.py train_unsloth.py eval_base.py` 后，Kernel → Restart & Run All。
- **kernel 无 torch（`ModuleNotFoundError: torch`）**：换 GPU PyTorch 内核（`conda_pytorch_p310`），或运行安装 cell 里的 `%pip install torch`。
- **`SFTConfig` / `SFTTrainer` 报未知参数**：TRL 版本差异，脚本已对 `processing_class/tokenizer` 做容错；如仍报错，`pip install -U trl` 到较新版。
- **`attn_implementation="sdpa"` 报错**：极老版 transformers 才有；升级 transformers 即可。
- **下载慢/失败**：设 `HF_ENDPOINT=https://hf-mirror.com` 或提前 `huggingface-cli download Qwen/Qwen3-0.6B`。
- **重启提醒**：装完 unsloth 没重启 kernel，可能出现库版本冲突——**重启后从第 1 步重跑**。